# Graph Neural Network Models 

TODO: Do more preprocessing (feature engineering) on the graph data

## Load

In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import DataLoader


import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [5]:
train_graph = torch.load('./data/processed_graph_train.pt', weights_only=False)
test_graph = torch.load('./data/processed_graph_test.pt', weights_only=False)

## Model 1: Basic Convolutional GNN

### Definition

##

In [6]:

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=16):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 2)  # 2 classes: root or not

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x


### Train

In [ ]:
# Load your preprocessed graph list
train_graphs = torch.load('./data/processed_graph_train.pt', weights_only=False)
train_loader = DataLoader(train_graphs, batch_size=1, shuffle=True)

# Infer input feature size (e.g., degree, centrality, etc.)
in_channels = train_graphs[0].x.size(1)

# Initialize model and optimizer
model = GCN(in_channels)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(20):
    total_loss = 0
    for data in train_loader:
        data = data.to('cpu')  # Or 'cuda' if using GPU
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)  # shape: [num_nodes, 2]
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} | Loss: {total_loss:.4f}')


/home/wtroiani/miniconda3/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 1 | Loss: 2462.3063
Epoch 2 | Loss: 2315.0864
Epoch 3 | Loss: 2231.2845
Epoch 4 | Loss: 2206.0269
Epoch 5 | Loss: 2105.6527
Epoch 6 | Loss: 2090.0529
Epoch 7 | Loss: 2086.7729
Epoch 8 | Loss: 2085.1328
Epoch 9 | Loss: 2078.2153


### Inference

In [ ]:
from torch_geometric.data import DataLoader

model.eval()
predicted_roots = []

# You may want to use a GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Load your test data
test_graphs = torch.load('./data/processed_graph_test.pt', weights_only=False)
test_loader = DataLoader(test_graphs, batch_size=1, shuffle=False)

with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        
        out = model(data.x, data.edge_index)  # Shape: [num_nodes, 2]
        pred = out.argmax(dim=1)              # Pick class with highest probability

        # Which node was predicted as root (class 1)?
        root_indices = (pred == 1).nonzero(as_tuple=True)[0]
        
        if len(root_indices) > 0:
            # If multiple, choose the one with highest confidence
            best_idx = root_indices[out[root_indices, 1].argmax()]
        else:
            # Fallback: pick node with highest predicted root probability
            best_idx = out[:, 1].argmax()

        # Map back to original node ID if stored
        original_node_id = data.node_ids[best_idx].item()
        predicted_roots.append({
            'language': data.language,
            'sentence_id': data.sentence_id,
            'predicted_root': original_node_id
        })


## Model 2: ???

### Definition

### Train

### Inference